# Gemma 4 E2B-IT Bio-Safety SAE Training â€” Colab T4

This notebook implements domain-specific fine-tuning for a TopK Sparse Autoencoder (SAE) on the Gemma 4 E2B-IT model. The goal is to enhance the SAE's representation of biosecurity concepts, specifically distinguishing between benign biological content and hazardous biological knowledge.

**Specifications:**
- **Model:** Gemma 4 E2B-IT (frozen, NF4 4-bit quantization).
- **SAE:** TopK (d_model=1536, d_sae=6144, k=32).
- **Loss:** L = L_recon + Î»_sparsity * L_sparsity + Î»_contrastive * L_contrastive.
- **Contrastive Objective:** Discriminates between hazard-adjacent and benign biological activations.

**Dataset options (cell [5/7]):**

| Option | Dataset | Tier coverage | Access |
|--------|---------|--------------|--------|
| `"wmdp"` (default) | `cais/wmdp-corpora` â€” WMDP Machine Unlearning Corpora (Li et al. 2024) | hazard + benign (~7.5K docs) | Public |
| `"gated"` | `SolshineCode/biorefusalaudit-gated` â€” BioRefusalAudit curated eval set | hazard + dual-use + benign (75â†’10K) | HF gated |
| `"local"` | Local `data/eval_set_public/eval_set_public_v1.jsonl` | all tiers (75 prompts) | Always available |

**Setup:**
1. Change runtime to **T4 GPU**.
2. Add `HF_TOKEN` (write scope) and `WANDB_API_KEY` to Colab ðŸ”‘ Secrets.
   - `HF_TOKEN` is required only for the `"gated"` dataset option and checkpoint saves to HF; not needed for the default `"wmdp"` path.
3. Toggle notebook access for both secrets.

In [ ]:
# [1/7] GPU + Environment Check
import torch
import subprocess

r = subprocess.run(["nvidia-smi"], capture_output=True, text=True)
print(r.stdout[:1000] if r.returncode == 0 else "NO GPU found.")

assert torch.cuda.is_available(), "CUDA not available â€” set Runtime to T4 GPU"
print("\nTorch version:", torch.__version__)
print("Device:", torch.cuda.get_device_name(0))
print("VRAM:", f"{torch.cuda.get_device_properties(0).total_memory/1e9:.2f} GB")

In [ ]:
# [2/7] Install Dependencies + Auth
# Install transformers from source: Gemma 4 ('gemma4' model type) requires the latest
# main branch. PyPI releases lag and raise KeyError: 'gemma4' on AutoConfig.
# Fix applied 2026-04-25 after KeyError: 'gemma4' on PyPI transformers install.
!pip install -q git+https://github.com/huggingface/transformers.git accelerate bitsandbytes datasets huggingface_hub wandb sentencepiece

import os, json, time
from google.colab import userdata
from huggingface_hub import login, HfApi
import wandb

for key in ("HF_TOKEN", "GITHUB_TOKEN", "WANDB_API_KEY"):
    try:
        os.environ[key] = userdata.get(key)
        print(f"{key}: loaded")
    except:
        print(f"{key}: not set")

if os.environ.get("HF_TOKEN"): login(token=os.environ["HF_TOKEN"], add_to_git_credential=False)
if os.environ.get("WANDB_API_KEY"): wandb.login(key=os.environ["WANDB_API_KEY"])

In [ ]:
# [2b/7] Repo Clone (optional â€” WMDP/HF-only path works without it)
#
# Priority:
#   1. GITHUB_TOKEN set â†’ authenticated clone (works for private repo)
#   2. No token, HF_TOKEN set â†’ unauthenticated clone attempt; if it fails,
#      continue in HF-only mode (WMDP data from HF, checkpoints to HF)
#   3. Neither token â†’ same unauthenticated attempt with same fallback
#
# REPO_AVAILABLE is set False when clone fails; cell [5/7] skips the
# "local" fallback and forces the WMDP or gated HF path instead.

import subprocess
from pathlib import Path

_REPO   = "SolshineCode/Deleeuw-AI-x-Bio-hackathon"
_BRANCH = "main"
_WORK   = Path("/content/Deleeuw-AI-x-Bio-hackathon")

_token = os.environ.get("GITHUB_TOKEN", "")
_clone_url = (
    f"https://{_token}@github.com/{_REPO}.git" if _token
    else f"https://github.com/{_REPO}.git"
)

REPO_AVAILABLE = False

if _WORK.exists():
    r = subprocess.run(
        ["git", "-C", str(_WORK), "pull", "--ff-only", "origin", _BRANCH],
        capture_output=True, text=True,
    )
    REPO_AVAILABLE = True  # dir exists â€” usable even if pull had nothing to do
    print("Repo present" + ("; pulled latest." if r.returncode == 0 else "; pull skipped (already up to date or detached HEAD)."))
else:
    r = subprocess.run(
        ["git", "clone", "--branch", _BRANCH, _clone_url, str(_WORK)],
        capture_output=True, text=True,
    )
    if r.returncode == 0:
        REPO_AVAILABLE = True
        print(f"Cloned {_REPO} @ {_BRANCH}.")
    else:
        hf_token = os.environ.get("HF_TOKEN", "")
        if hf_token:
            print(f"Clone failed (rc={r.returncode}) â€” HF_TOKEN is set, continuing in HF-only mode.")
            print("Dataset: WMDP from HuggingFace.  Checkpoints: uploaded to your HF account.")
        else:
            print(f"Clone failed (rc={r.returncode}) and no HF_TOKEN set.")
            print("Add GITHUB_TOKEN or HF_TOKEN to Colab Secrets (ðŸ”‘) and re-run.")
        print(f"git stderr: {r.stderr.strip()[:300]}")

if REPO_AVAILABLE:
    %cd /content/Deleeuw-AI-x-Bio-hackathon
    print("Working directory: /content/Deleeuw-AI-x-Bio-hackathon")
else:
    print("Working directory: /content  (HF-only mode)")

In [ ]:
# [3/7] Load Frozen Gemma 4 E2B-IT
import torch.nn as nn
from transformers import AutoTokenizer, BitsAndBytesConfig

HF_MODEL = "google/gemma-4-E2B-it"
LAYER = 17
D_MODEL = 1536

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16
)

print(f"Loading {HF_MODEL}...")
# Gemma 4 is a multimodal model; try CausalLM first, fall back to ImageTextToText
try:
    from transformers import AutoModelForCausalLM
    model = AutoModelForCausalLM.from_pretrained(
        HF_MODEL, quantization_config=bnb_config,
        device_map={"":0}, low_cpu_mem_usage=True
    )
except Exception:
    from transformers import AutoModelForImageTextToText
    model = AutoModelForImageTextToText.from_pretrained(
        HF_MODEL, quantization_config=bnb_config,
        device_map={"":0}, low_cpu_mem_usage=True
    )
model.eval()
tokenizer = AutoTokenizer.from_pretrained(HF_MODEL)
tokenizer.pad_token = tokenizer.eos_token

class ActivationHook:
    def __init__(self): self.last = None
    def __call__(self, module, inputs, outputs):
        # Overwrite (not append) to avoid VRAM accumulation during generation
        self.last = outputs[0].detach() if isinstance(outputs, tuple) else outputs.detach()

def pick_layer(m, layer_idx):
    """Find transformer block list across Gemma 2/3/4 text-only & multimodal layouts.

    Tries common attribute paths first, then falls back to a named_modules() scan
    so a transformers-version refactor (e.g. Gemma 4's model.language_model.layers
    layout) doesn't break activation hooking.
    """
    # Fast path: known layouts in order of likelihood
    for path in (
        "model.language_model",   # Gemma 4 ForConditionalGeneration (current)
        "language_model.model",   # Gemma 3 ForConditionalGeneration (older)
        "language_model",         # some multimodal variants
        "model",                  # text-only ForCausalLM
        "transformer",            # GPT-style fallback
    ):
        obj = m
        try:
            for attr in path.split("."):
                obj = getattr(obj, attr)
        except AttributeError:
            continue
        if hasattr(obj, "layers"):
            return obj.layers[layer_idx]
    # Slow path: scan all submodules for a ModuleList named "layers"
    for name, mod in m.named_modules():
        if name.endswith(".layers") and hasattr(mod, "__getitem__"):
            try:
                return mod[layer_idx]
            except (IndexError, TypeError):
                continue
    raise AttributeError(
        f"Could not locate transformer layer list in {type(m).__name__}; "
        f"top-level children: {[n for n, _ in m.named_children()]}"
    )

hook = ActivationHook()
handle = pick_layer(model, LAYER).register_forward_hook(hook)
print(f"Attached hook at layer {LAYER}")


In [ ]:
# [4/7] TopKSAE Architecture
import torch.nn as nn

# D_MODEL is set in [3/7] (Gemma 4 E2B hidden size = 1536)
D_SAE = 6144   # 4x expansion ratio
K     = 32     # TopK sparsity matching Gemma Scope convention

class TopKSAE(nn.Module):
    """TopK Sparse Autoencoder.

    forward(x) -> (x_hat, z, pre_relu):
      x_hat    - reconstruction (L_recon)
      z        - sparse feature activations (L_contrastive)
      pre_relu - pre-activation logits (L_sparsity L1 penalty)
    """
    def __init__(self, d_model, d_sae, k):
        super().__init__()
        self.d_model, self.d_sae, self.k = d_model, d_sae, k
        self.W_enc = nn.Parameter(torch.randn(d_model, d_sae) * 0.02)
        self.b_enc = nn.Parameter(torch.zeros(d_sae))
        self.W_dec = nn.Parameter(torch.randn(d_sae, d_model) * 0.02)
        self.b_dec = nn.Parameter(torch.zeros(d_model))
        self.normalize_decoder()

    def encode(self, x):
        pre = x @ self.W_enc + self.b_enc
        pre_relu = torch.relu(pre)
        topk_vals, topk_idx = torch.topk(pre_relu, self.k, dim=-1)
        out = torch.zeros_like(pre_relu)
        out.scatter_(-1, topk_idx, topk_vals)
        return out

    def decode(self, z):
        return z @ self.W_dec + self.b_dec

    def forward(self, x):
        pre = x @ self.W_enc + self.b_enc
        z   = self.encode(x)
        x_hat = self.decode(z)
        return x_hat, z, pre

    @torch.no_grad()
    def normalize_decoder(self):
        n = self.W_dec.norm(dim=1, keepdim=True).clamp_min(1e-8)
        self.W_dec.data.div_(n)

    @torch.no_grad()
    def project_grad(self):
        if self.W_dec.grad is None:
            return
        g, w = self.W_dec.grad, self.W_dec.data
        g.sub_((g * w).sum(dim=1, keepdim=True) * w)


sae = TopKSAE(D_MODEL, D_SAE, K).to('cuda').float()
print(f'SAE: d_model={D_MODEL}, d_sae={D_SAE}, k={K}')
print(f'Parameters: {sum(p.numel() for p in sae.parameters()):,}')


In [ ]:
# [5/7] Configurable Dataset Loading
from datasets import load_dataset, Dataset as HFDataset, concatenate_datasets

# ======================================================================
# DATASET CONFIGURATION
# ======================================================================
# Option A (DEFAULT): cais/wmdp-corpora â€” WMDP Machine Unlearning Corpora
#   (Li et al. 2024, arXiv:2403.03218). Publicly available on HuggingFace.
#   bio_forget_corpus â†’ hazard_adjacent_category (N~3.9K docs)
#   bio_retain_corpus â†’ benign_bio (N~3.7K docs)
#   Structural fit: these are the exact paired corpora designed to contrast
#   hazardous vs. safe biology knowledge â€” ideal for contrastive SAE training.
#   Works without GITHUB_TOKEN â€” data comes entirely from HuggingFace.
#
# Option B: SolshineCode/biorefusalaudit-gated â€” curated 75-prompt eval set
#   with native BioRefusalAudit tier labels. Requires HF approval (gated).
#   Use this once you have access to the full institutional corpus (~10K).
#
# Option C: local â€” reads data/eval_set_public/eval_set_public_v1.jsonl from
#   the cloned repo. Only available if REPO_AVAILABLE=True (cell [2b/7]).
#   Auto-demoted to "wmdp" when REPO_AVAILABLE=False.
#
# Set DATASET_SOURCE = "wmdp" | "gated" | "local"
# ======================================================================

DATASET_SOURCE = "wmdp"   # <-- change to "gated" once biorefusalaudit-gated is accessible

# Guard: local option requires the cloned repo
if DATASET_SOURCE == "local" and not REPO_AVAILABLE:
    print("NOTE: DATASET_SOURCE='local' requested but repo clone failed â€” switching to 'wmdp'.")
    print("WMDP path works without GITHUB_TOKEN; HF checkpoints will be uploaded to your account.")
    DATASET_SOURCE = "wmdp"

HF_TEXT_COLUMN  = "prompt"
HF_LABEL_COLUMN = "tier"

dataset = None

if DATASET_SOURCE == "wmdp":
    # NOTE 2026-04-25: cais/wmdp-corpora bio_forget_corpus is NOT publicly available.
    # bio-retain-corpus (benign, ~3.7K docs) IS available. We try both hazard config
    # name forms (underscore/hyphen) and fall back gracefully to benign-only training.
    # This demonstrates the institutional data gap documented in paper Â§8.
    ds_hazard = None
    ds_benign = None
    for hazard_cfg in ("bio-forget-corpus", "bio_forget_corpus"):
        try:
            _h = load_dataset("cais/wmdp-corpora", hazard_cfg, split="train")
            ds_hazard = _h.map(lambda x: {"prompt": x["text"], "tier": "hazard_adjacent_category"},
                               remove_columns=_h.column_names)
            print(f"Hazard corpus ({hazard_cfg}): {len(ds_hazard)} docs")
            break
        except Exception as _e:
            print(f"Hazard corpus '{hazard_cfg}' unavailable: {_e}")
    for benign_cfg in ("bio-retain-corpus", "bio_retain_corpus"):
        try:
            _b = load_dataset("cais/wmdp-corpora", benign_cfg, split="train")
            ds_benign = _b.map(lambda x: {"prompt": x["text"], "tier": "benign_bio"},
                               remove_columns=_b.column_names)
            print(f"Benign corpus ({benign_cfg}): {len(ds_benign)} docs")
            break
        except Exception as _e:
            print(f"Benign corpus '{benign_cfg}' unavailable: {_e}")
    if ds_benign is None:
        print("WMDP benign corpus unavailable; switching to local/synthetic fallback.")
        DATASET_SOURCE = "local"
    else:
        MAX_PER_TIER = 5000
        ds_benign = ds_benign.select(range(min(MAX_PER_TIER, len(ds_benign))))
        if ds_hazard is not None:
            ds_hazard = ds_hazard.select(range(min(MAX_PER_TIER, len(ds_hazard))))
            dataset = concatenate_datasets([ds_hazard, ds_benign]).shuffle(seed=42)
            print(f"Dataset: {len(ds_hazard)} hazard + {len(ds_benign)} benign = {len(dataset)} total")
        else:
            print("WARNING: bio hazard corpus not publicly available. Training benign-only.")
            print("L_contrastive will be 0. See paper Â§8 for institutional access path.")
            dataset = ds_benign
            print(f"Dataset: 0 hazard + {len(ds_benign)} benign = {len(dataset)} total (benign-only)")
        HF_TEXT_COLUMN, HF_LABEL_COLUMN = "prompt", "tier"

elif DATASET_SOURCE == "gated":
    HF_DATASET_REPO = "SolshineCode/biorefusalaudit-gated"
    try:
        print(f"Loading {HF_DATASET_REPO}...")
        dataset = load_dataset(HF_DATASET_REPO, split="train")
        print(f"Loaded {len(dataset)} examples.")
    except Exception as e:
        print(f"Gated dataset unavailable ({e}); falling back to local eval set.")
        DATASET_SOURCE = "local"

if DATASET_SOURCE == "local" or dataset is None:
    # No gate on REPO_AVAILABLE: fall back to synthetic data rather than crashing
    if REPO_AVAILABLE:
        import json as _json
        _rows = [_json.loads(l) for l in open("data/eval_set_public/eval_set_public_v1.jsonl")]
        dataset = HFDataset.from_list(_rows)
        HF_TEXT_COLUMN, HF_LABEL_COLUMN = "prompt", "tier"
        print(f"Using local public eval set: {len(dataset)} examples.")
    else:
        # Synthetic smoke-test data â€” no external access needed
        _synthetic = (
            [{"prompt": f"How do pathogens cause disease? Q{i}", "tier": "benign_bio"} for i in range(20)] +
            [{"prompt": f"What are biosafety containment levels? Q{i}", "tier": "hazard_adjacent_category"} for i in range(10)]
        )
        dataset = HFDataset.from_list(_synthetic)
        HF_TEXT_COLUMN, HF_LABEL_COLUMN = "prompt", "tier"
        print(f"Synthetic fallback: {len(dataset)} examples (smoke-test only).")
def sample_batch(ds, size=4):
    indices = torch.randint(0, len(ds), (size,))
    return [ds[int(i)] for i in indices]

In [ ]:
import torch.nn.functional as F

# [6/7] Training Loop Logic
def compute_contrastive_loss(z_hazard, z_benign, margin=1.0):
    """Encourage separation between hazard-adjacent and benign biological feature profiles."""
    if z_hazard.size(0) == 0 or z_benign.size(0) == 0: return torch.tensor(0.0).to(z_hazard.device)
    
    # Aggregate across sequence tokens
    m_hazard = z_hazard.mean(0)
    m_benign = z_benign.mean(0)
    
    # Minimize similarity between the two profiles
    cos_sim = F.cosine_similarity(m_hazard.unsqueeze(0), m_benign.unsqueeze(0))
    return cos_sim.mean()

def track_semantic_drift(sae, tokenizer, features_to_watch=[100, 500, 1000]):
    """Placeholder for semantic drift tracking: identifies top tokens for specific features."""
    # In a real run, this would collect activations across a calibration set.
    return {f"feat_{idx}_drift": 0.0 for idx in features_to_watch}

In [ ]:
# [7/7] Main Execution Loop
from tqdm.auto import tqdm

RUN_TAG = "gemma4-e2b-bio-sae-v1"
HF_OUTPUT_REPO = f"Solshine/{RUN_TAG}"
LR = 3e-4
LAMBDA_SPARSITY = 0.04
LAMBDA_CONTRASTIVE = 0.1
BATCH_SIZE = 4
MAX_STEPS = 2000
SAVE_EVERY = 500

optimizer = torch.optim.AdamW(sae.parameters(), lr=LR)

# If no WANDB_API_KEY, run in disabled mode to avoid interactive hang.
_wandb_mode = "online" if os.environ.get("WANDB_API_KEY") else "disabled"
wandb.init(project="biorefusalaudit-sae-training", name=RUN_TAG, mode=_wandb_mode, config={
    "model": HF_MODEL, "layer": LAYER, "d_sae": D_SAE, "k": K,
    "lr": LR, "lambda_sparsity": LAMBDA_SPARSITY, "lambda_contrastive": LAMBDA_CONTRASTIVE
})

pbar = tqdm(range(MAX_STEPS))
for step in pbar:
    batch = sample_batch(dataset, BATCH_SIZE)
    
    all_activations = []
    hazard_indices = []
    benign_indices = []
    offset = 0
    
    for item in batch:
        prompt = item[HF_TEXT_COLUMN]
        tier = item.get(HF_LABEL_COLUMN, "benign_bio")

        # Wrap with Gemma chat template so the RLHF safety circuit engages (writeup Â§4.5
        # pass-5 finding: without canonical <start_of_turn>/<end_of_turn> tokens, Gemma 4
        # IT receives input outside its training distribution, refusal-circuit features
        # don't fire, and the resulting SAE features won't transfer to deployment regime).
        # WMDP-corpora documents are wrapped as user messages too â€” uniform input regime
        # beats mixing chat-formatted eval prompts with raw corpus text.
        if getattr(tokenizer, "chat_template", None):
            msgs = [{"role": "user", "content": prompt}]
            formatted = tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
        else:
            formatted = prompt
        inputs = tokenizer(formatted, return_tensors="pt").to("cuda")
        with torch.no_grad():
            # Single forward pass: captures all prompt token activations at once.
            # model.generate() fires the hook once per new token (32 times),
            # so hook.last would only hold the last generated token's activation.
            _ = model(**inputs)
        
        acts = hook.last # (1, seq, d_model)
        seq_len = acts.size(1)
        all_activations.append(acts.squeeze(0))
        
        token_range = range(offset, offset + seq_len)
        if "hazard" in tier: hazard_indices.extend(token_range)
        elif "benign" in tier: benign_indices.extend(token_range)
        offset += seq_len
        
    # Move activations to GPU/Float32 for SAE
    x = torch.cat(all_activations, dim=0).cuda().float()
    x_hat, z, pre_relu = sae(x)
    
    # Loss 1: Reconstruction (MSE)
    l_recon = F.mse_loss(x_hat, x)
    
    # Loss 2: Sparsity (L1 on pre-activations)
    l_sparsity = pre_relu.abs().mean()
    
    # Loss 3: Contrastive (Tier separation)
    l_contrast = torch.tensor(0.0).cuda()
    if hazard_indices and benign_indices:
        l_contrast = compute_contrastive_loss(z[hazard_indices], z[benign_indices])
        
    total_loss = l_recon + LAMBDA_SPARSITY * l_sparsity + LAMBDA_CONTRASTIVE * l_contrast
    
    optimizer.zero_grad()
    total_loss.backward()
    sae.project_grad()
    optimizer.step()
    sae.normalize_decoder()
    
    # Metrics
    l0 = (z > 0).float().sum(-1).mean().item()
    drift = track_semantic_drift(sae, tokenizer)
    
    log_dict = {
        "step": step,
        "total_loss": total_loss.item(),
        "l_recon": l_recon.item(),
        "l_sparsity": l_sparsity.item(),
        "l_contrastive": l_contrast.item(),
        "l0": l0
    }
    log_dict.update(drift)
    wandb.log(log_dict)
    
    pbar.set_postfix({"loss": f"{total_loss.item():.4f}", "L0": f"{l0:.1f}"})
    
    if (step + 1) % SAVE_EVERY == 0:
        torch.save(sae.state_dict(), "sae_weights.pt")
        try:
            api = HfApi()
            api.create_repo(HF_OUTPUT_REPO, exist_ok=True, private=True)
            api.upload_file(path_or_fileobj="sae_weights.pt", 
                            path_in_repo=f"sae_weights_step_{step+1}.pt", 
                            repo_id=HF_OUTPUT_REPO)
            print(f"\nCheckpoint saved to HF: {HF_OUTPUT_REPO}")
        except Exception as e: print(f"\nHF Push failed: {e}")

handle.remove()
wandb.finish()

# Final checkpoint -- upload regardless of step alignment with SAVE_EVERY
torch.save(sae.state_dict(), "sae_weights_final.pt")
if os.environ.get("HF_TOKEN"):
    try:
        api = HfApi()
        api.create_repo(HF_OUTPUT_REPO, exist_ok=True, private=True)
        api.upload_file(
            path_or_fileobj="sae_weights_final.pt",
            path_in_repo="sae_weights_final.pt",
            repo_id=HF_OUTPUT_REPO
        )
        print(f"Final checkpoint saved to: https://huggingface.co/{HF_OUTPUT_REPO}")
    except Exception as e:
        print(f"Final HF upload failed: {e}")
else:
    print("No HF_TOKEN -- final weights saved locally as sae_weights_final.pt")

print("Training Complete.")